In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Load Required Tables
# ============================================================

subscription_cdc_df = spark.table(
    "customer360_dev.control.subscription_cdc_work"
)

silver_df = spark.table(
    "customer360_dev.silver.subscriptions"
)

bronze_df = spark.table(
    "customer360_dev.bronze.subscriptions"
)

In [0]:
run_id_row = (
    subscription_cdc_df
    .select("_pipeline_run_id")
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .first()
)

run_id = (
    run_id_row["_pipeline_run_id"]
    if run_id_row
    else None
)

print("Reconciling Subscription Run ID:", run_id)

Reconciling Subscription Run ID: None


In [0]:
if run_id is not None:

    current_bronze_df = (
        bronze_df
        .filter(
            F.col("_pipeline_run_id") == run_id
        )
    )

else:

    current_bronze_df = bronze_df.limit(0)

    print(
        "No subscription incremental run "
        "available for reconciliation."
    )

No subscription incremental run available for reconciliation.


In [0]:
insert_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .count()
)

update_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .count()
)

unchanged_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "UNCHANGED")
    .count()
)

classified_count = (
    insert_count
    + update_count
    + unchanged_count
)

print(f"INSERT     : {insert_count:,}")
print(f"UPDATE     : {update_count:,}")
print(f"UNCHANGED  : {unchanged_count:,}")
print(f"CLASSIFIED : {classified_count:,}")

INSERT     : 0
UPDATE     : 0
UNCHANGED  : 0
CLASSIFIED : 0


In [0]:
source_distinct_subscriptions = (
    current_bronze_df
    .select("subscription_id")
    .distinct()
    .count()
)

classification_difference = (
    source_distinct_subscriptions
    - classified_count
)

print(
    f"Distinct Bronze subscriptions : "
    f"{source_distinct_subscriptions:,}"
)

print(
    f"Classified subscriptions      : "
    f"{classified_count:,}"
)

print(
    f"Difference                    : "
    f"{classification_difference:,}"
)

Distinct Bronze subscriptions : 0
Classified subscriptions      : 0
Difference                    : 0


In [0]:
insert_missing_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .select("subscription_id")
    .join(
        silver_df.select("subscription_id"),
        on="subscription_id",
        how="left_anti"
    )
    .count()
)

print(
    "INSERT subscriptions missing from Silver:",
    insert_missing_count
)

INSERT subscriptions missing from Silver: 0


In [0]:
update_missing_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .select("subscription_id")
    .join(
        silver_df.select("subscription_id"),
        on="subscription_id",
        how="left_anti"
    )
    .count()
)

print(
    "UPDATE subscriptions missing from Silver:",
    update_missing_count
)

UPDATE subscriptions missing from Silver: 0


In [0]:
duplicate_subscription_ids = (
    silver_df
    .groupBy("subscription_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate subscription IDs in Silver:",
    duplicate_subscription_ids
)

Duplicate subscription IDs in Silver: 0


In [0]:
orphan_subscriptions = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        ).select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Orphan subscriptions in Silver:",
    orphan_subscriptions
)

Orphan subscriptions in Silver: 0


In [0]:
reconciliation_pass = (
    classification_difference == 0
    and insert_missing_count == 0
    and update_missing_count == 0
    and duplicate_subscription_ids == 0
    and orphan_subscriptions == 0
)

status = (
    "PASS"
    if reconciliation_pass
    else "FAIL"
)

print(
    "Subscription CDC Reconciliation:",
    status
)

Subscription CDC Reconciliation: PASS


In [0]:
result = [
    {
        "entity": "subscriptions",
        "pipeline_run_id": run_id,
        "source_distinct_subscriptions": source_distinct_subscriptions,
        "insert_count": insert_count,
        "update_count": update_count,
        "unchanged_count": unchanged_count,
        "classified_count": classified_count,
        "classification_difference": classification_difference,
        "insert_missing_in_silver": insert_missing_count,
        "update_missing_in_silver": update_missing_count,
        "duplicate_subscription_ids": duplicate_subscription_ids,
        "orphan_subscriptions": orphan_subscriptions,
        "status": status
    }
]

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)

schema = StructType([
    StructField("entity", StringType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("source_distinct_subscriptions", LongType(), True),
    StructField("insert_count", LongType(), True),
    StructField("update_count", LongType(), True),
    StructField("unchanged_count", LongType(), True),
    StructField("classified_count", LongType(), True),
    StructField("classification_difference", LongType(), True),
    StructField("insert_missing_in_silver", LongType(), True),
    StructField("update_missing_in_silver", LongType(), True),
    StructField("duplicate_subscription_ids", LongType(), True),
    StructField("orphan_subscriptions", LongType(), True),
    StructField("status", StringType(), True),
])

display(
    spark.createDataFrame(result, schema)
)

entity,pipeline_run_id,source_distinct_subscriptions,insert_count,update_count,unchanged_count,classified_count,classification_difference,insert_missing_in_silver,update_missing_in_silver,duplicate_subscription_ids,orphan_subscriptions,status
subscriptions,null,0,0,0,0,0,0,0,0,0,0,PASS
